## SGLD Deep Energy-based Model Example
This notebook provides an example of a deep EBM using Stochastic Gradient Lagevin Dynamics to train the model. The model is implemented using PyTorch, rather than Keras / Tensorflow which is used more frequently in the book. This notebook is based on [Du and Mordatch (2019)](https://papers.nips.cc/paper/2019/file/378a063b8fdb1db941e34f4bde584c7d-Paper.pdf) and [Lippe (2021)](https://uvadlc-notebooks.readthedocs.io/en/latest/tutorial_notebooks/tutorial8/Deep_Energy_Models.html) (see comments below).

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
import multiprocessing
import math
import os
import random

In [ ]:
## PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.data as data
import torch.optim as optim
# Torchvision
import torchvision
from torchvision.datasets import MNIST
from torchvision import transforms
from torchvision.utils import make_grid, save_image
# PyTorch Lightning
import pytorch_lightning as pl
from pytorch_lightning.callbacks import LearningRateMonitor, ModelCheckpoint
from pytorch_lightning.loggers import CSVLogger

In [ ]:
## Pandas
import pandas as pd

In [ ]:
# Configure environment
# Ensure that all operations are deterministic on GPU (if used) for reproducibility
torch.backends.cudnn.determinstic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")
print("Device:", device)

### Load and Prepare MNIST Dataset
MNIST is also a built in dataset with torchvision and we load it here and prepare dataloaders for PyTorch. Note that for this model values are scaled to be between -1 and 1, rather than between 0 and 1 as in previous examples.

In [ ]:
transform = transforms.Compose([transforms.ToTensor(),
                                 transforms.Normalize((0.5,), (0.5,))
                               ])

mnist_trainset = MNIST(root='./data', train=True, download=True, transform=transform)
mnist_testset = MNIST(root='./data', train=False, download=True, transform=transform)

In [ ]:
train_batch_size = 128
test_batch_size = 256
num_workers = 4
train_loader = data.DataLoader(mnist_trainset, batch_size=train_batch_size, 
                               shuffle=True,  drop_last=True,  
                               num_workers=num_workers, pin_memory=True)
test_loader  = data.DataLoader(mnist_testset,  batch_size=test_batch_size, 
                               shuffle=False, drop_last=False, 
                               num_workers=num_workers)

In [ ]:
def show_tensor_images(image_tensor, 
                       num_images=64, 
                       size=(1, 32, 32), 
                       nrow=8, 
                       filename=""):
    image_unflat = image_tensor.detach().cpu().view(-1, *size)
    image_grid = make_grid(image_unflat[:num_images], nrow=nrow)
    img = image_grid.permute(1, 2, 0).squeeze()
    plt.axis('off')
    plt.imshow(img)
    if len(filename) > 0:
        plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.show()

### Define CNN model for Energy
We model the energy function as a CNN, given it takes an MNIST image as input and returns the energy scalar.

In [ ]:
class Swish(nn.Module):

    def forward(self, x):
        return x * torch.sigmoid(x)

In [ ]:
class CNNModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=5, stride=2, padding=4),
            Swish(),
            nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1),
            Swish(),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            Swish(),
            nn.Conv2d(64, 64, kernel_size=3, stride=2, padding=1),
            Swish(),
            nn.Flatten(),
            nn.Linear(256, 64),
            Swish(),
            nn.Linear(64, 1)
        )
        
    def forward(self, x):
        x = self.layers(x).squeeze(dim=-1)
        return x

### Define Sampler Class
The sampler class manages the sampling process using *Stochastic Gradient Langevin Dynamics (SGLD)*. The model adopts the approach of Du and Mordatch (2019)  with its 'replay buffer' which is used to initialize samples 95% of the time. In practice to get good samples from Lagevin dynamics requires quite a lot of steps, which would slow learning considerably if pure random initialisation was used. The use of a replay buffer means that samples are mostly initialised using examples stored previously which allows fewer steps to be used during training.

In [ ]:
class Sampler:

    def __init__(self, model, img_shape, sample_size, buffer_pct=0.95, max_len=8192):
        super().__init__()
        self.model = model
        self.img_shape = img_shape
        self.sample_size = sample_size
        self.max_len = max_len
        self.buffer_pct = buffer_pct
        self.examples = [(torch.rand((1,)+img_shape, device=device)*2-1) \
                         for _ in range(self.sample_size)]

    def sample_new_examples(self, steps=60, step_size=10):
        # Choose buffer_pct% of the batch from the buffer, the rest randomly
        n_new = np.random.binomial(self.sample_size, 1.0-self.buffer_pct)
        rand_imgs = torch.rand((n_new,) + self.img_shape, device=device)*2-1
        if self.buffer_pct > 0.0:
            select_imgs = torch.cat(random.choices(self.examples, 
                                               k=self.sample_size-n_new), dim=0)
            input_imgs = torch.cat([rand_imgs, select_imgs], dim=0).detach()
        else:
            input_imgs = rand_imgs.detach()

        # Perform MCMC sampling
        input_imgs = Sampler.generate_samples(self.model, input_imgs, 
                                              steps=steps, step_size=step_size)

        # Add new images to the buffer and remove old ones if needed
        self.examples = list(input_imgs.chunk(self.sample_size, dim=0)) \
                        + self.examples
        self.examples = self.examples[:self.max_len]
        return input_imgs

    @staticmethod
    def generate_samples(model, input_imgs, steps=60, step_size=10, 
                         return_img_per_step=False):
        # Switch off unnecessary gradients
        is_training = model.training
        model.eval()
        for p in model.parameters():
            p.requires_grad = False
        input_imgs.requires_grad = True

        # Enable gradient calculation if not already the case
        had_gradients_enabled = torch.is_grad_enabled()
        torch.set_grad_enabled(True)

        noise = torch.randn(input_imgs.shape, device=device)

        # List for storing generations at each step
        imgs_per_step = []

        # Loop over K (steps)
        for _ in range(steps):
            # Add noise to the input and clip to range[-1,1]
            noise.normal_(0, 0.005)
            input_imgs.data.add_(noise.data)
            input_imgs.data.clamp_(min=-1.0, max=1.0)

            # Calculate gradients for the current input.
            # Note that by convention the neural network models -Energy
            # Hence here a minus sign is needed
            out_en = -model(input_imgs)
            out_en.sum().backward()
            # clip the gradients
            input_imgs.grad.data.clamp_(min=-0.03, max=0.03) 

            # Complete the Lagevin sampling update
            input_imgs.data.add_(-step_size * input_imgs.grad.data)
            input_imgs.grad.detach_()
            
            
            input_imgs.grad.zero_()
            input_imgs.data.clamp_(min=-1.0, max=1.0)

            if return_img_per_step:
                imgs_per_step.append(input_imgs.clone().detach())

        # Switch model gradients back on
        for p in model.parameters():
            p.requires_grad = True
        model.train(is_training)

        # Reset gradient calculation to setting before this function
        torch.set_grad_enabled(had_gradients_enabled)

        if return_img_per_step:
            return torch.stack(imgs_per_step, dim=0)
        else:
            return input_imgs

### Define Deep Energy Model
Here a deep energy model class is defined that provides overrides of training and validation steps

In [ ]:
class DeepEnergyModel(pl.LightningModule):
    def __init__(self, img_shape, batch_size, 
                 alpha=0.1, lr=1e-4, beta1=0.0, 
                 buffer_pct=0.95,
                 no_steps=60):
        super().__init__()
        self.save_hyperparameters()
        self.cnn = CNNModel()
        self.sampler = Sampler(self.cnn, img_shape=img_shape, 
                               sample_size=batch_size,
                               buffer_pct=buffer_pct)
        self.example_input_array = torch.zeros(1, *img_shape)
        self.no_steps = no_steps

    def forward(self, x):
        z = self.cnn(x)
        return z

    def configure_optimizers(self):
        # Note that the default here is to use Adam without momentum 
        # as EBMs can have problems with momentum.
        optimizer = optim.Adam(self.parameters(), 
                               lr=self.hparams.lr, 
                               betas=(self.hparams.beta1, 0.999))
        scheduler = optim.lr_scheduler.StepLR(optimizer, 1, gamma=0.97)
        return [optimizer], [scheduler]

    def training_step(self, batch, batch_idx):
        # Add a small amount of noise
        real_imgs, _ = batch
        small_noise = torch.randn_like(real_imgs) * 0.005
        real_imgs.add_(small_noise).clamp_(min=-1.0, max=1.0)

        # Obtain samples
        fake_imgs = self.sampler.sample_new_examples(steps=self.no_steps, step_size=10)

        # Predict energy score for all images
        inp_imgs = torch.cat([real_imgs, fake_imgs], dim=0)
        pos_en, neg_en = self.cnn(inp_imgs).chunk(2, dim=0)

        # Calculate losses
        reg_loss = self.hparams.alpha * (pos_en ** 2 + neg_en ** 2).mean()
        # Note cdiv_loss is negative - positive to account for sign of Energy function
        cdiv_loss = neg_en.mean() - pos_en.mean()
        loss = reg_loss + cdiv_loss

        # Logging
        self.log('loss', loss)
        self.log('loss_l2', reg_loss)
        self.log('loss_cd', cdiv_loss)
        self.log('avg_en_pos', pos_en.mean())
        self.log('avg_en_neg', neg_en.mean())
        return loss

    def validation_step(self, batch, batch_idx):
        # For validating, calculate the contrastive divergence 
        # between purely random images and unseen examples
        real_imgs, _ = batch
        fake_imgs = torch.rand_like(real_imgs)*2-1

        inp_imgs = torch.cat([real_imgs, fake_imgs], dim=0)
        pos_en, neg_en = self.cnn(inp_imgs).chunk(2, dim=0)

        cdiv = neg_en.mean() - pos_en.mean()
        self.log('val_cd', cdiv)
        self.log('val_en_pos', pos_en.mean())
        self.log('val_en_neg', neg_en.mean())

### Train the Model

In [ ]:
class GenerateCallback(pl.Callback):
    def __init__(self, filepath, batch_size=8, vis_steps=8, num_steps=256, every_n_epochs=5):
        super().__init__()
        self.batch_size = batch_size
        self.vis_steps = vis_steps
        self.num_steps = num_steps
        self.filepath = filepath
        self.every_n_epochs = every_n_epochs

    def on_train_epoch_end(self, trainer, pl_module):
        if trainer.current_epoch % self.every_n_epochs == 0:
            imgs_per_step = self.generate_imgs(pl_module)
            step_size = self.num_steps // self.vis_steps

            for i in range(imgs_per_step.shape[1]):
                imgs_to_plot = imgs_per_step[step_size - 1::step_size, i]
                grid = torchvision.utils.make_grid(
                    imgs_to_plot,
                    nrow=imgs_to_plot.shape[0],
                    normalize=True,
                    value_range=(-1, 1),
                )
                filename = os.path.join(self.filepath, f"generation_{i}.png")
                torchvision.utils.save_image(grid, filename)

    def generate_imgs(self, pl_module):
        pl_module.eval()
        start_imgs = torch.rand(
            (self.batch_size,) + pl_module.hparams["img_shape"],
            device=device
        ) * 2 - 1

        torch.set_grad_enabled(True)
        imgs_per_step = Sampler.generate_samples(
            pl_module.cnn,
            start_imgs,
            steps=self.num_steps,
            step_size=10,
            return_img_per_step=True,
        )
        torch.set_grad_enabled(False)
        pl_module.train()
        return imgs_per_step

In [ ]:
def train_model(checkpoint_path, logger, max_epochs, **kwargs):
    # Create a PyTorch Lightning trainer with the generation callback
    rootdir = os.path.join(checkpoint_path, "MNIST")
    trainer = pl.Trainer(default_root_dir=rootdir,
                         accelerator="gpu" if str(device).startswith("cuda") else "cpu",
                        devices=1,
                         max_epochs=max_epochs,
                         gradient_clip_val=0.1,
                         callbacks=[ModelCheckpoint(save_weights_only=True, 
                                                    save_top_k=-1, 
                                                    every_n_epochs=5),
                                    GenerateCallback(rootdir, every_n_epochs=5),
                                    LearningRateMonitor("epoch")
                                   ],
                        logger=logger)
    pl.seed_everything(101)
    model = DeepEnergyModel(**kwargs)
    trainer.fit(model, train_loader, test_loader)
    return model

In [ ]:
checkpoint_path = './EBM/check_point/'
checkpoint_path_mnist = './EBM/check_point/MNIST'
os.makedirs(checkpoint_path, exist_ok=True)
os.makedirs(checkpoint_path_mnist, exist_ok=True)
max_epochs = 60

In [ ]:
logger = CSVLogger("logs", name="MNIST_EBM")
model = train_model(checkpoint_path,
                    logger,
                    max_epochs,
                    img_shape=(1,28,28),
                    batch_size=train_loader.batch_size,
                    lr=1e-4,
                    beta1=0.0)

### Sample from the Model


In [ ]:
model.to(device)
pl.seed_everything(10001)
filepath = 'dummy'
callback = GenerateCallback(filepath, batch_size=4, vis_steps=8, num_steps=256)
imgs_per_step = callback.generate_imgs(model)
imgs_per_step = imgs_per_step.cpu()

In [ ]:
for i in range(imgs_per_step.shape[1]):
    step_size = callback.num_steps // callback.vis_steps
    imgs_to_plot = imgs_per_step[step_size-1::step_size,i]
    imgs_to_plot = torch.cat([imgs_per_step[0:1,i],imgs_to_plot], dim=0)
    grid = torchvision.utils.make_grid(imgs_to_plot, 
                                       nrow=imgs_to_plot.shape[0], 
                                       normalize=True, value_range=(-1,1), 
                                       pad_value=0.5, padding=2)
    grid = grid.permute(1, 2, 0)
    plt.figure(figsize=(8,8))
    plt.imshow(grid)
    plt.xlabel("Generation iteration")
    plt.xticks([(imgs_per_step.shape[-1]+2)*(0.5+j) for j in range(callback.vis_steps+1)],
               labels=[1] + list(range(step_size,imgs_per_step.shape[0]+1,step_size)))
    plt.yticks([])
    plt.savefig('SGLDGenerationIt_' + str(i) + '.png', dpi=300, bbox_inches='tight')

In [ ]:
model.to(device)
pl.seed_everything(103)
callback = GenerateCallback(filepath, batch_size=40, vis_steps=1, num_steps=1024)
imgs_per_step = callback.generate_imgs(model)

In [ ]:
plt.figure(figsize=(10,20))
show_tensor_images(imgs_per_step[1023,:,:,:],
                   num_images=40, 
                   size=(1, 28, 28), 
                   nrow=20,  
                   filename='SGLDGenerationFinal.png')

### Approximate Nijkamp et al. Model
[Nijkamp et al (2019)](https://arxiv.org/abs/1904.09770) adopted a similar approach to Du and Mordatch (2019), however this initialised the samples randomly each time and used 100 steps of SGLD during training. The model can be reconfigured to use this approach by setting the buffer percentage to 0.0 and the number of steps to 100 in the Sampler class.

In [ ]:
checkpoint_path = './EBMNijkamp/check_point/'
checkpoint_path_mnist = './EBMNijkamp/check_point/MNIST'
os.makedirs(checkpoint_path, exist_ok=True)
os.makedirs(checkpoint_path_mnist, exist_ok=True)
max_epochs = 60

In [ ]:
logger = CSVLogger("logs", name="MNIST_EBM_Nijkamp")
model_nijkamp = train_model(checkpoint_path,
                    logger,
                    max_epochs,
                    img_shape=(1,28,28),
                    batch_size=train_loader.batch_size,
                    lr=1e-4,
                    beta1=0.0,
                    buffer_pct=0.0,
                    no_steps=100)

In [ ]:
model_nijkamp.to(device)
pl.seed_everything(9999)
filepath = 'dummy'
callback = GenerateCallback(filepath, batch_size=4, vis_steps=8, num_steps=256)
imgs_per_step = callback.generate_imgs(model_nijkamp)
imgs_per_step = imgs_per_step.cpu()

In [ ]:
for i in range(imgs_per_step.shape[1]):
    step_size = callback.num_steps // callback.vis_steps
    imgs_to_plot = imgs_per_step[step_size-1::step_size,i]
    imgs_to_plot = torch.cat([imgs_per_step[0:1,i],imgs_to_plot], dim=0)
    grid = torchvision.utils.make_grid(imgs_to_plot, 
                                       nrow=imgs_to_plot.shape[0], 
                                       normalize=True, value_range=(-1,1), 
                                       pad_value=0.5, padding=2)
    grid = grid.permute(1, 2, 0)
    plt.figure(figsize=(8,8))
    plt.imshow(grid)
    plt.xlabel("Generation iteration")
    plt.xticks([(imgs_per_step.shape[-1]+2)*(0.5+j) for j in range(callback.vis_steps+1)],
               labels=[1] + list(range(step_size,imgs_per_step.shape[0]+1,step_size)))
    plt.yticks([])
    plt.savefig('SGLDGenerationItNijkamp_' + str(i) + '.png', dpi=300, bbox_inches='tight')

In [ ]:
model_nijkamp.to(device)
pl.seed_everything(103)
callback = GenerateCallback(filepath, batch_size=40, vis_steps=1, num_steps=1024)
imgs_per_step = callback.generate_imgs(model_nijkamp)

In [ ]:
plt.figure(figsize=(10,20))
show_tensor_images(imgs_per_step[1023,:,:,:],
                   num_images=40, 
                   size=(1, 28, 28), 
                   nrow=20,  
                   filename='SGLDGenerationNijkampFinal.png')

Note that this code is derived from the work of Phillip Lippe which is distributed under the following licence:


MIT License

Copyright (c) 2020 Phillip Lippe

Permission is hereby granted, free of charge, to any person obtaining a copy of this software and associated documentation files (the "Software"), to deal in the Software without restriction, including without limitation the rights to use, copy, modify, merge, publish, distribute, sublicense, and/or sell copies of the Software, and to permit persons to whom the Software is furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY, FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM, OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE SOFTWARE.